# M7 Verification: Repository Inactivity

M7 is a descriptive repository-inactivity diagnostic for RQ3. It does not identify planning omission or measure off-repository work.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import plotly.express as px
from IPython.display import Image, display
ROOT=Path.cwd()
while ROOT != ROOT.parent and not (ROOT/'paper_v9').is_dir(): ROOT=ROOT.parent
METRICS=ROOT/'paper_v9'/'data'/'metrics'
FIGURES=ROOT/'paper_v9'/'figures'; FIGURES.mkdir(parents=True, exist_ok=True)
sys.path.insert(0,str(ROOT))
trajectory=pd.read_csv(METRICS/'m7_inactivity_trajectory.csv',dtype={'Semestre':str})
checkpoint=pd.read_csv(METRICS/'m7_checkpoint_inactivity.csv',dtype={'Semestre':str})
pattern=pd.read_csv(METRICS/'m7_inactivity_pattern.csv',dtype={'Semestre':str})
metadata=json.loads((METRICS/'m7_repository_inactivity.metadata.json').read_text())
legacy=pd.read_csv(ROOT/'paper_v8'/'data'/'m7_planning_omission_rate.csv',dtype={'Semestre':str})

## 1. Verification

The official inputs are Git commit timestamps and evaluator vote timestamps. The anchor is the latest evaluator vote per team and checkpoint.

In [ ]:
paper=(ROOT/'paper_v8'/'latex_code'/'main.tex').read_text(encoding='utf-8')
assert paper.index(r'\subsubsection{RQ3:') < paper.index(r'\textbf{M7 --')
assert metadata['rq']=='RQ3'
assert metadata['planning_claim']=='not_identifiable_from_repository_inactivity'
assert metadata['window_days']==7
assert metadata['coverage']=={'team_semesters':14,'checkpoint_rows':42,'daily_rows':994,'pattern_rows':14}
assert len(trajectory)==994 and len(checkpoint)==42 and len(pattern)==14
assert trajectory.groupby(['ID_Equipe','Semestre']).size().eq(71).all()
assert set(checkpoint['temporal_marker'])=={'T1','T2','T3'}
assert trajectory['commit_n_7d'].ge(0).all()
assert trajectory['repository_inactive_7d'].isin([True,False]).all()
print('M7 RQ3, anchor, window, coverage, and interpretation contract: PASS')

## 2. V8 to V9 traceability

The legacy omission rate is retained only for audit. M7 v9 replaces it with repository-inactivity trajectories and never treats missing Git activity as proof of missing planning.

In [ ]:
traceability=pd.DataFrame([
{'v8_recommendation':'Do not interpret missing T1 score as planning omission.','v9_decision':'Replace omission rate with repository_inactivity outputs.','status':'applied','evidence':'m7_repository_inactivity.metadata.json','limitation_or_approval':'Git cannot observe off-repository planning.'},
{'v8_recommendation':'Measure inactivity dynamically around checkpoints.','v9_decision':'Use seven-day retrospective windows anchored to evaluator votes.','status':'applied','evidence':'m7_checkpoint_inactivity.csv','limitation_or_approval':'Windows overlap.'},
{'v8_recommendation':'Distinguish persistent from temporary inactivity.','v9_decision':'Publish patterns and T3-relative daily trajectory.','status':'applied','evidence':'m7_inactivity_pattern.csv and trajectory.csv','limitation_or_approval':'Git activity only.'},
{'v8_recommendation':'Do not duplicate M6a as a planning predictor.','v9_decision':'Exclude M7 from M9 predictor inputs.','status':'applied','evidence':'metadata limitations','limitation_or_approval':'Secondary diagnostic.'},
])
assert set(traceability['status'])=={'applied'}
display(traceability)
display(legacy)

## 3. Artifact demo

The figures consume the official M7 trajectory and checkpoint CSVs.

In [ ]:
cohort=(trajectory.groupby(['Semestre','window_end_day_relative_to_t3'],as_index=False).agg(team_n=('ID_Equipe','size'),inactive_team_n=('repository_inactive_7d','sum'),recent_commit_n=('commit_n_7d','sum')))
cohort['repository_inactivity_rate']=cohort['inactive_team_n']/cohort['team_n']
figure=px.line(cohort,x='window_end_day_relative_to_t3',y='repository_inactivity_rate',color='Semestre',markers=True,title='M7 repository inactivity around T3')
figure.add_vline(x=0,line_dash='dash')
figure.write_html(METRICS/'m7_repository_inactivity_trajectory.html',include_plotlyjs='cdn')
for extension in ('pdf','svg','png'): figure.write_image(FIGURES/f'm7_repository_inactivity_trajectory.{extension}',scale=2 if extension=='png' else 1)
for extension in ('pdf','svg','png'): assert (FIGURES/f'm7_repository_inactivity_trajectory.{extension}').is_file() and (FIGURES/f'm7_repository_inactivity_trajectory.{extension}').stat().st_size>0
assert (METRICS/'m7_repository_inactivity_trajectory.html').stat().st_size>0
print('M7 HTML, PDF, SVG, and PNG figures generated: PASS')

In [ ]:
display(checkpoint.head(6))
display(pattern)
display(cohort.loc[cohort['window_end_day_relative_to_t3'].isin([-63,-21,-7,0,7])])
display(Image(filename=str(FIGURES/'m7_repository_inactivity_trajectory.png'),width=900))
print('M7 artifact demo: official tables and figure displayed')

## Preliminary RQ3 analysis

M7 contributes a descriptive timing and coverage view of repository activity. It can distinguish recent inactivity patterns around T3, but it cannot establish planning omission, project quality, effort, or causality. M7 is not an independent predictor for M9.

In [ ]:
summary=cohort.groupby('Semestre',as_index=False).agg(team_semester_windows=('team_n','sum'),mean_inactivity_rate=('repository_inactivity_rate','mean'),min_inactivity_rate=('repository_inactivity_rate','min'),max_inactivity_rate=('repository_inactivity_rate','max'))
summary['analysis_level']='team_semester_rolling_window_then_checkpoint_pattern'
summary['inference']='descriptive_only'
summary['planning_claim']='not_identifiable_from_repository_inactivity'
assert summary['inference'].eq('descriptive_only').all()
assert summary['planning_claim'].eq('not_identifiable_from_repository_inactivity').all()
display(summary)
print('Preliminary RQ3 reading: M7 describes repository inactivity only; no planning or causal claim is supported.')